In [2]:
import pandas as pd
from glob import glob

files = glob("../data/202112/*.csv.gz")

for f in files:
    print("=" * 80)
    print(f"FILE: {f}")
    df = pd.read_csv(f, compression="gzip", nrows=5)
    display(df)


# First analysis: important files
- Flights: FACT TABLE

### Another files I might use later:
- Flight_Points_*: Measure route efficiency
- Flight_FIRs_*: FIR means airspace crossed. Estimate workload per airspace.
- Flight_AUAs_* → ATC units, measure delay per ATC, see if there are any patterns.

In [4]:


flights = pd.read_csv("../data/raw/flights/Flights_20211201_20211231.csv.gz", compression="gzip", nrows=5)
flights.head()

,ECTRL ID,ADEP,ADEP Latitude,ADEP Longitude,ADES,ADES Latitude,ADES Longitude,FILED OFF BLOCK TIME,FILED ARRIVAL TIME,ACTUAL OFF BLOCK TIME,ACTUAL ARRIVAL TIME,AC Type,AC Operator,AC Registration,ICAO Flight Type,STATFOR Market Segment,Requested FL,Actual Distance Flown (nm)
0,248113105,KORD,41.98000,-87.90500,EGLL,51.47750,-0.46139,01-12-2021 00:00:00,01-12-2021 07:36:15,01-12-2021 00:12:00,01-12-2021 07:45:12,B772,AAL,N779AN,S,Mainline,310,3588
1,248115421,ENKB,63.11194,7.82611,EDDV,52.46028,9.68361,01-12-2021 00:00:00,01-12-2021 01:42:23,30-11-2021 23:56:00,01-12-2021 01:39:35,C560,ZZZ,DCAPB,N,Business Aviation,410,654
2,248117757,LTFM,41.27528,28.75194,HLLM,32.89444,13.27778,01-12-2021 00:00:00,01-12-2021 04:00:17,01-12-2021 00:10:00,01-12-2021 04:10:49,A319,LWA,5AWLA,S,Mainline,350,1299
3,248120507,ESSA,59.65194,17.91861,ESNU,63.79306,20.28000,01-12-2021 00:00:00,01-12-2021 01:21:56,01-12-2021 00:05:39,01-12-2021 01:26:05,AT75,AZD,HBALR,N,All-Cargo,150,267
4,248120508,KHPN,41.06667,-73.71667,EGKB,51.33083,0.03250,01-12-2021 00:00:00,01-12-2021 06:39:50,01-12-2021 00:02:00,01-12-2021 06:34:01,GLEX,PVA,N926PN,N,Business Aviation,410,3069


# Analysis of Flights table
## Field per field
ECTRL ID: Unique EUROCONTROL flight identifier; primary key used to join with other flight-related tables.

ADEP: ICAO code of the departure airport.

ADEP Latitude: Latitude of the departure airport.

ADEP Longitude: Longitude of the departure airport.

ADES: ICAO code of the arrival airport.

ADES Latitude: Latitude of the arrival airport.

ADES Longitude: Longitude of the arrival airport.

FILED OFF BLOCK TIME: Scheduled off-block (pushback) time.

FILED ARRIVAL TIME: Scheduled arrival time at destination.

ACTUAL OFF BLOCK TIME: Actual off-block (pushback) time.

ACTUAL ARRIVAL TIME: Actual arrival time at destination.

AC Type: Aircraft type according to ICAO classification.

AC Operator: ICAO code of the airline or operator.

AC Registration: Aircraft registration identifier.

ICAO Flight Type: ICAO classification of the flight.

STATFOR Market Segment: Market segment classification defined by EUROCONTROL STATFOR.

Requested FL: Requested cruising flight level.

Actual Distance Flown (nm): Total distance actually flown, expressed in nautical miles.

### More data added- csvs for ICAO
airlines
airport
ac
### include weather?
i have tried several ways of downloading files with all airports weather info, no luck.
use next code for each airport

# Schema
### Fact table
Flights (main events: departure, arrival, flight ID, times, distances, aircraft, operator…)

### Dimension tables (ICAO references)
Airports → ICAO → name, country, coordinates

Aircraft types → ICAO → model, engine type

Airlines → ICAO → name

### Enrichment (i have to see if i use them)
Weather → ICAO + timestamp → METAR/TAF conditions (optional, i have to see)

Flight_Points_* → trajectory: compute actual trajectory lengths, segment speeds, and detect unusual climbs/descents.

Flight_FIRs_* → airspace: analyze which FIRs are most congested and compare filed vs actual FIR crossings.

Flight_AUAs_* → ATC units: examine actual vs filed ATC handovers and time spent per unit.

# Flights analysis

In [5]:
flights = pd.read_csv("../data/raw/flights/Flights_20211201_20211231.csv.gz", compression='gzip', parse_dates=[
    "FILED OFF BLOCK TIME",
    "FILED ARRIVAL TIME",
    "ACTUAL OFF BLOCK TIME",
    "ACTUAL ARRIVAL TIME"
])

flights.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 570200 entries, 0 to 570199
Data columns (total 18 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   ECTRL ID                    570200 non-null  int64  
 1   ADEP                        570200 non-null  object 
 2   ADEP Latitude               569936 non-null  float64
 3   ADEP Longitude              569936 non-null  float64
 4   ADES                        570200 non-null  object 
 5   ADES Latitude               569879 non-null  float64
 6   ADES Longitude              569879 non-null  float64
 7   FILED OFF BLOCK TIME        570200 non-null  object 
 8   FILED ARRIVAL TIME          570200 non-null  object 
 9   ACTUAL OFF BLOCK TIME       570200 non-null  object 
 10  ACTUAL ARRIVAL TIME         570200 non-null  object 
 11  AC Type                     570200 non-null  object 
 12  AC Operator                 570200 non-null  object 
 13  AC Registratio

In [6]:
flights.describe()


,ECTRL ID,ADEP Latitude,ADEP Longitude,ADES Latitude,ADES Longitude,Requested FL,Actual Distance Flown (nm)
count,5.702000e+05,569936.000000,569936.000000,569879.000000,569879.000000,570087.000000,570200.000000
mean,2.484266e+08,45.486004,9.532935,45.469511,9.720354,315.137165,1063.136719
std,1.802146e+05,10.801538,24.855432,10.845841,24.733481,79.819897,1269.689474
min,2.481131e+08,-34.838330,-157.917500,-34.857220,-123.183890,0.000000,0.000000
25%,2.482702e+08,40.472220,-0.190280,40.472220,-0.190280,280.000000,311.000000
50%,2.484283e+08,47.458060,8.570560,47.458060,8.570560,340.000000,581.000000
75%,2.485846e+08,51.885000,19.784720,51.885000,19.912220,370.000000,1153.000000
max,2.487351e+08,78.246110,140.378330,78.246110,140.378330,490.000000,9282.000000


In [35]:
# Number of unique airports, airlines, aircraft types
print("Unique departure airports:", flights['ADEP'].nunique())
print("Unique arrival airports:", flights['ADES'].nunique())
print("Unique aircraft types:", flights['AC Type'].nunique())
print("Unique airlines:", flights['AC Operator'].nunique())

# Most frequent routes
flights.groupby(['ADEP', 'ADES']).size().sort_values(ascending=False).head(10)



Unique departure airports: 1401
Unique arrival airports: 1400
Unique aircraft types: 220
Unique airlines: 574


ADEP  ADES
ENGM  ENVA    706
ENVA  ENGM    706
LEBL  LEPA    604
GCLP  GCXO    602
ENBR  ENGM    599
LEPA  LEBL    595
GCXO  GCLP    595
ENGM  ENBR    592
LFPO  LFMN    548
LFMN  LFPO    545
dtype: int64

In [9]:
time_cols = [
    'FILED OFF BLOCK TIME',
    'FILED ARRIVAL TIME',
    'ACTUAL OFF BLOCK TIME',
    'ACTUAL ARRIVAL TIME'
]

for col in time_cols:
    flights[col] = pd.to_datetime(flights[col], errors='coerce', dayfirst=True)

# Flight duration (actual) in minutes
flights['duration_actual_min'] = (flights['ACTUAL ARRIVAL TIME'] - flights['ACTUAL OFF BLOCK TIME']).dt.total_seconds() / 60

# Departure / arrival delays in minutes
flights['dep_delay_min'] = (flights['ACTUAL OFF BLOCK TIME'] - flights['FILED OFF BLOCK TIME']).dt.total_seconds() / 60
flights['arr_delay_min'] = (flights['ACTUAL ARRIVAL TIME'] - flights['FILED ARRIVAL TIME']).dt.total_seconds() / 60

# Quick stats
flights[['duration_actual_min','dep_delay_min','arr_delay_min']].describe()


,duration_actual_min,dep_delay_min,arr_delay_min
count,570200.000000,570200.000000,570200.000000
mean,165.938202,2.773608,2.270981
std,159.431355,14.223965,16.810209
min,0.066667,-460.450000,-814.383333
25%,69.383333,-5.000000,-7.000000
50%,105.900000,1.000000,0.716667
75%,184.000000,8.000000,9.287500
max,1150.366667,945.816667,946.383333


In [37]:
# By airline
flights.groupby('AC Operator')['dep_delay_min'].mean().sort_values(ascending=False).head(10)

# By aircraft type
flights.groupby('AC Type')['arr_delay_min'].mean().sort_values(ascending=False).head(10)

# By airport
flights.groupby('ADEP')['dep_delay_min'].mean().sort_values(ascending=False).head(10)


ADEP
OSLK    136.416667
KSAF    101.000000
UADD     95.450000
CYEG     74.285714
KBIF     72.666667
CYMX     66.894203
ZSYT     60.094444
WBGG     57.633333
CYYC     55.315982
KMEM     54.235714
Name: dep_delay_min, dtype: float64

In [38]:
flights.isna().sum()

ECTRL ID                         0
ADEP                             0
ADEP Latitude                  264
ADEP Longitude                 264
ADES                             0
ADES Latitude                  321
ADES Longitude                 321
FILED OFF BLOCK TIME             0
FILED ARRIVAL TIME               0
ACTUAL OFF BLOCK TIME            0
ACTUAL ARRIVAL TIME              0
AC Type                          0
AC Operator                      0
AC Registration               1446
ICAO Flight Type                 0
STATFOR Market Segment           0
Requested FL                   113
Actual Distance Flown (nm)       0
duration_actual_min              0
dep_delay_min                    0
arr_delay_min                    0
dtype: int64